In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from pyfaidx import Fasta
import regex
from sklearn.model_selection import train_test_split
import os
%matplotlib inline
pd.options.mode.chained_assignment = None 
np.random.seed = 42

We considered 325 of confirmed ORIs, none longer than 500 bp, as positive instances to meet DNABERT’s
input size requirement. Shorter ORIs were asymmetrically extended with real genomic
flanking nucleotides of the corresponding origin sequence, to obtain sequences of uniform
length (500 bp). The required extension length was randomly allocated between
the left and right sides of the ORIs, while ensuring that the extended regions did not
exceed the boundaries of the chromosome or overlap with neighboring annotated ORI.

In [2]:
# read oridb originssgd_
yeast_origins_oridb = pd.read_csv("/p/project/hai_dnaori/piroozeh1/yeast-origins/data/yeast_oridb_confirmed_origins.csv")
yeast_origins_oridb["length"] = yeast_origins_oridb.end - yeast_origins_oridb.start + 1
yeast_origins_oridb

,chr,start,end,name,length
0,1,650,1791,"102,00 ARS",1142
1,1,6136,7136,ARS102.5,1001
2,1,7998,8548,"103,00 ARS",551
3,1,30946,31184,"104,00 ARS",239
4,1,40716,43300,"105,00 ARS",2585
...,...,...,...,...,...
405,16,776921,777152,ARS1626.5,232
406,16,819153,819393,"1627,00 ARS",241
407,16,842646,842894,"1628,00 ARS",249
408,16,880854,881102,"1630,00 ARS",249


In [3]:
 # filter out origins > 500 bp length (limitation by DNABert)
yeast_origins_oridb_filtered = yeast_origins_oridb[yeast_origins_oridb.length <= 500  ]
yeast_origins_oridb_filtered.to_csv("/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/yeast_oridbLess500.csv", index=False)
yeast_origins_oridb_filtered

,chr,start,end,name,length
3,1,30946,31184,"104,00 ARS",239
5,1,70258,70491,"106,00 ARS",234
6,1,124350,124599,"107,00 ARS",250
9,1,159906,160127,"109,00 ARS",222
10,1,176154,176402,"110,00 ARS",249
...,...,...,...,...,...
405,16,776921,777152,ARS1626.5,232
406,16,819153,819393,"1627,00 ARS",241
407,16,842646,842894,"1628,00 ARS",249
408,16,880854,881102,"1630,00 ARS",249


In [4]:
 # randomly pick start and end indices to get a total window length of 500 contatining the origin
yeast_origins_oridb_filtered["delta"] = 500 - yeast_origins_oridb_filtered.length
yeast_origins_oridb_filtered["pre"] = yeast_origins_oridb_filtered.apply(lambda x: 0 if x.delta==0 else np.random.randint(0, x.delta), axis=1)
yeast_origins_oridb_filtered["post"] = yeast_origins_oridb_filtered.delta - yeast_origins_oridb_filtered.pre
yeast_origins_oridb_filtered

,chr,start,end,name,length,delta,pre,post
3,1,30946,31184,"104,00 ARS",239,261,94,167
5,1,70258,70491,"106,00 ARS",234,266,156,110
6,1,124350,124599,"107,00 ARS",250,250,237,13
9,1,159906,160127,"109,00 ARS",222,278,239,39
10,1,176154,176402,"110,00 ARS",249,251,88,163
...,...,...,...,...,...,...,...,...
405,16,776921,777152,ARS1626.5,232,268,260,8
406,16,819153,819393,"1627,00 ARS",241,259,26,233
407,16,842646,842894,"1628,00 ARS",249,251,138,113
408,16,880854,881102,"1630,00 ARS",249,251,235,16


In [5]:
    # apply new start and end indices
yeast_origins_oridb_filtered_new = yeast_origins_oridb_filtered.copy()
yeast_origins_oridb_filtered_new.start = yeast_origins_oridb_filtered_new.start - yeast_origins_oridb_filtered_new.pre
yeast_origins_oridb_filtered_new.end = yeast_origins_oridb_filtered_new.end + yeast_origins_oridb_filtered_new.post
yeast_origins_oridb_filtered_new.drop(columns=["delta", "pre", "post", "length", "name"], inplace=True)
yeast_origins_oridb_filtered_new["label"] = 1
yeast_origins_oridb_filtered_new

,chr,start,end,label
3,1,30852,31351,1
5,1,70102,70601,1
6,1,124113,124612,1
9,1,159667,160166,1
10,1,176066,176565,1
...,...,...,...,...
405,16,776661,777160,1
406,16,819127,819626,1
407,16,842508,843007,1
408,16,880619,881118,1


In [6]:
# make sure that there is no start outside the chromosome
yeast_origins_oridb_filtered_new.loc[yeast_origins_oridb_filtered_new.start < 0, "end"] -= yeast_origins_oridb_filtered_new.loc[yeast_origins_oridb_filtered_new.start < 0].start - 1
yeast_origins_oridb_filtered_new.loc[yeast_origins_oridb_filtered_new.start < 0, "start"] -= yeast_origins_oridb_filtered_new.loc[yeast_origins_oridb_filtered_new.start < 0].start - 1


To create 325 non-origin sequences, negative instances
were randomly chosen from parts of the genome that did not overlap with any origin
sequences. This approach
ensures a diverse representation of non-origin sequences derived from real genomic
data.

In [7]:
# loop over all chromosomes and sample windows of length 500 that don't intersect with origns
negative_windows = []
for chr, origins in yeast_origins_oridb_filtered_new.groupby("chr"):
    # collect all blocked indices (origins plus already samples windows)
    blocked_indices = []
    for index, origin in origins.iterrows():
        blocked_indices.append([*range(origin.start, origin.end)])
    blocked_indices = np.hstack(blocked_indices)
    # read chromosome to detect its max length
    fasta = Fasta(f"/p/project/hai_dnaori/sgd_data/S288C_reference_sequence_R64-3-1_20210421.fsa")
    chr_len = len(fasta[f"chr{chr}"])
    num_negatives_yet = len(negative_windows)
    # create as many negative windows as positives per origin
    while len(negative_windows) < num_negatives_yet + len(origins):
        # pick a random start and compute the following 500 indices
        random_start = np.random.randint(0, chr_len - 500)
        window_indices = range(random_start, random_start + 500)
        # check if those indices intersects with the blocked ones
        if len(np.intersect1d(blocked_indices, window_indices)) == 0:
            negative_windows.append([chr, random_start, random_start+499, 0])
            # add new indices to blocked ones so that we have no intersecting windows
            blocked_indices = np.append(blocked_indices, window_indices)

In [8]:
full_dataset = yeast_origins_oridb_filtered_new.append(pd.DataFrame(negative_windows, columns=yeast_origins_oridb_filtered_new.columns))
full_dataset.sort_values(["chr", "start"], inplace=True)
full_dataset


,chr,start,end,label
3,1,30852,31351,1
1,1,37635,38134,0
4,1,40225,40724,0
2,1,46350,46849,0
3,1,58156,58655,0
...,...,...,...,...
324,16,887725,888224,0
308,16,890506,891005,0
310,16,898949,899448,0
320,16,929353,929852,0


In [ ]:

# save indices to csv
#Randomly extended sequnce Window=500
full_dataset.to_csv("/p/project/hai_dnaori/piroozeh1/yeast-origins/data/window_dataset_indices_oridb.csv", index=False)

In [31]:
# read sequence data
sequence_data = Fasta(f"/p/project/hai_dnaori/sgd_data/S288C_reference_sequence_R64-3-1_20210421.fsa")

# add the DNA sequence to each region
full_dataset["seq"] = full_dataset.apply(lambda x: sequence_data[f"chr{x.chr}"][x.start-1: x.end].seq, axis=1)
full_dataset



,chr,start,end,label,seq
2,1,3534,4033,0,TTGCATTGATAAAGTACCTACTATCCCAGACTATATTTGTATACAA...
1,1,5948,6447,0,TGTAAACCAGTGGATTTTTGCTCAACATATAAAAAACTAAAGACCT...
3,1,17686,18185,0,TTCCATTGGGGTTAGTAGCAGGATATATATTTCCATTCTATATTCC...
3,1,30781,31280,1,ATATTTTAATGTTAAGATGAAATTTAAGTGAGCTGGTAATATCAAG...
0,1,36766,37265,0,TAAGAGGGGCAAGAAGTTAAATAAAAAGGCTCTGGAAGACAAACTG...
...,...,...,...,...,...
408,16,880682,881181,1,GGATTGTCAAGACACTCCGGTATTACTCGAGCCCGTAATACAACAG...
304,16,901761,902260,0,TCAGGAACTTGGTTTTCAACCCTACTGCATTGTTTCCCACGCGGCG...
313,16,905376,905875,0,TGCAAAGACTTTGAAGCATGGTTTAATTCCAAACTTGCTGGATGCC...
314,16,932083,932582,0,GAGGTACTATGCCGTATACATTAAATGTGTAACAAGTTTGGATACC...


In [2]:
def seq2kmer(seq, k):
        """
        Convert original sequence to kmers
        
        Arguments:
        seq -- str, original sequence.
        k -- int, kmer of length k specified.
        
        Returns:
        kmers -- str, kmers separated by space

        """
        kmer = [seq[x:x+k] for x in range(len(seq)+1-k)]
        kmers = " ".join(kmer)
        return kmers

    # add kmers with k = 4
# full_dataset["sequence"] = full_dataset.apply(lambda x: seq2kmer(x.seq, 4), axis=1)
# full_dataset

In [3]:
def add_chr_prefix(chromosome):
    return f'chr{chromosome}'

In [ ]:
#fetch and prepare train dataset represented by range
# generate 7 data splits using random_state= 42, 25, 30, 50, 60, 70, 100
# train test split
data_path_DNABER="DNABERT"
data_path_DNABER2="DNABERT2"
window_dataset = pd.read_csv("/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/AllData/window_dataset_indices_oridb.csv")

sequence_data = Fasta(f"/p/project1/hai_dnaori/sgd_data/S288C_reference_sequence_R64-3-1_20210421.fsa")

# add the DNA sequence to each region
window_dataset["seq"] = window_dataset.apply(lambda x: sequence_data[f"chr{x.chr}"][x.start-1: x.end].seq, axis=1)
#add kmer
window_dataset["sequence"] = window_dataset.apply(lambda x: seq2kmer(x.seq, 4), axis=1)
# train test split
X_train, X_test, y_train, y_test = train_test_split(window_dataset[["chr", "start", "end", "seq", "kmer"]], window_dataset.label, test_size=0.2, random_state=100, stratify=window_dataset.label)

# train valid split
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.125, random_state=100, stratify=y_train)
train_new = pd.DataFrame({"sequence": X_train.seq, "label": y_train.values})
valid_new = pd.DataFrame({"sequence": X_valid.seq, "label": y_valid.values})
test_new = pd.DataFrame({"sequence": X_test.seq, "label": y_test.values})

train_new_kmer = pd.DataFrame({"sequence": X_train.kmer, "label": y_train.values})
valid_new_kmer = pd.DataFrame({"sequence": X_valid.kmer, "label": y_valid.values})
test_new_kmer = pd.DataFrame({"sequence": X_test.kmer, "label": y_test.values})

folder= "train_dev_test100"
# save final kmer files to tsv
train_new_kmer.to_csv(os.path.join(data_path_DNABER, folder, "4/train.tsv"), sep="\t", index=False)
valid_new_kmer.to_csv(os.path.join(data_path_DNABER, folder,"4/dev.tsv"), sep="\t", index=False)
test_new_kmer.to_csv(os.path.join(data_path_DNABER, folder,"4/test.tsv"), sep="\t", index=False)

# save final files to tsv
train_new.to_csv(os.path.join(data_path_DNABER,folder, "train.tsv"), sep="\t", index=False)
valid_new.to_csv(os.path.join(data_path_DNABER,folder, "dev.tsv"), sep="\t", index=False)
test_new.to_csv(os.path.join(data_path_DNABER,folder, "test.tsv"), sep="\t", index=False)

#DNABERT2
train_new.to_csv(os.path.join(data_path_DNABER2,folder, "train.csv"), index=False)
valid_new.to_csv(os.path.join(data_path_DNABER2,folder, "dev.csv"), index=False)
test_new.to_csv(os.path.join(data_path_DNABER2,folder, "test.csv"), index=False)





train_window = pd.DataFrame({"chr": X_train.chr, "start": X_train.start, "end": X_train.end,"seq": X_train.seq, "label": y_train.values})
valid_window = pd.DataFrame({"chr": X_valid.chr, "start": X_valid.start, "end": X_valid.end,"seq": X_valid.seq, "label": y_valid.values})
test_window = pd.DataFrame({"chr": X_test.chr, "start": X_test.start, "end": X_test.end,"seq": X_test.seq, "label": y_test.values})


# train_window.drop(columns=["seq"], inplace=True)
train_window.to_csv(os.path.join(data_path_DNABER,folder,  "train_window.tsv"), sep="\t", index=False)
valid_window.to_csv(os.path.join(data_path_DNABER,folder,  "dev_window.tsv"), sep="\t", index=False)
test_window.to_csv(os.path.join(data_path_DNABER,folder,  "test_window.tsv"), sep="\t", index=False)
# # 
train_window.drop(columns=["seq"], inplace=True)
valid_window.drop(columns=["seq"], inplace=True)
test_window.drop(columns=["seq"], inplace=True)

train_window.to_csv(os.path.join(data_path_DNABER,folder,  "train_window.bed"),sep='\t', header=False, index=False)
valid_window.to_csv(os.path.join(data_path_DNABER,folder,  "dev_window.bed"),sep='\t', header=False, index=False)
test_window.to_csv(os.path.join(data_path_DNABER,folder,  "test_window.bed"),sep='\t', header=False, index=False)

